# Training and Baseline Evaluation

In [ ]:
%matplotlib inline
import sys
import os
import torch
import numpy as np
import matplotlib.subplots as plt
import matplotlib.pyplot as plt

# Route to project root AND change working directory
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)
    
# This is the magic line that fixes the FileNotFoundError!
os.chdir(project_root) 

from src.sdk.sdk import SineDenoisingSDK

print("Initializing Sine Denoising System via SDK...")
sdk = SineDenoisingSDK()

# 1. Generate Data & Train
# ... (Keep the rest of Cell 1 exactly the same below this line)

print("Initializing Sine Denoising System via SDK...")
sdk = SineDenoisingSDK()


Initializing Sine Denoising System via SDK...


FileNotFoundError: Config file not found: config/setup.json

In [ ]:

# 1. Generate Data & Train
sdk.initialize_dataset()
sdk.train_models(epochs=50)


In [ ]:

# 2. Get Final Metrics
print("\nFinal Model Evaluation:")
report = sdk.get_evaluation_report()
for model, mse in report.items():
    print(f"{model}: {mse:.6f}")


In [ ]:

# 3. Visualizing a Single Sample
mlp_x_sample = sdk.data["mlp_x"][0:1]
rnn_x_sample = sdk.data["rnn_x"][0:1]
target_sample = sdk.data["targets"][0].cpu().numpy()
noisy_sample = sdk.data["mlp_x"][0][:10].cpu().numpy()

results = sdk.execute_denoising(mlp_x_sample, rnn_x_sample)

plt.figure(figsize=(12, 6))
plt.plot(noisy_sample, label="Noisy Input", linestyle=":", color="gray")
plt.plot(target_sample, label="Pure Target", linewidth=3, color="black")
plt.plot(results["mlp_out"][0].cpu().numpy(), label="MLP Denoised")
plt.plot(results["rnn_out"][0].cpu().numpy(), label="RNN Denoised")
plt.plot(results["lstm_out"][0].cpu().numpy(), label="LSTM Denoised")
plt.title("Signal Denoising: Model Predictions vs Truth")
plt.legend()
plt.show()


In [ ]:

# 4. Bar Chart Comparison
plt.figure(figsize=(8, 5))
plt.bar(report.keys(), report.values(), color=['blue', 'orange', 'green'])
plt.title("Mean Squared Error (MSE) Comparison")
plt.ylabel("MSE (Lower is Better)")
plt.show()

# Sensitivity Analysis

In [ ]:
# Sensitivity Analysis: How does Amplitude Noise affect performance?
noise_levels = [0.01, 0.05, 0.1, 0.2, 0.5]
mlp_mses, rnn_mses, lstm_mses = [], [], []

print("Running Sensitivity Analysis...")
for noise in noise_levels:
    test_sdk = SineDenoisingSDK()
    test_sdk.config.sigma_amplitude = noise
    test_sdk.config.num_samples = 1000 # Smaller dataset for speed
    test_sdk.initialize_dataset()
    test_sdk.train_models(epochs=20)
    
    rep = test_sdk.get_evaluation_report()
    mlp_mses.append(rep["MLP_MSE"])
    rnn_mses.append(rep["RNN_MSE"])
    lstm_mses.append(rep["LSTM_MSE"])

plt.figure(figsize=(10, 6))
plt.plot(noise_levels, mlp_mses, marker='o', label="MLP")
plt.plot(noise_levels, rnn_mses, marker='s', label="RNN")
plt.plot(noise_levels, lstm_mses, marker='^', label="LSTM")
plt.title("Sensitivity Analysis: Noise vs. Error")
plt.xlabel("Amplitude Noise Level (Sigma)")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()